# DC Motor PID Control Analysis
## CASE A: No Disturbing Load 
#### Data Import

In [21]:
import pandas as pd  
import numpy as np   

file_path = 'data/noload_DC_motor.csv'

noload = pd.read_csv(file_path)
noload

,time,theta_PID1_noload,theta_PID2_noload,theta_PID3_noload,theta_ref,Volt_PID1_noload,Volt_PID2_noload,Volt_PID3_noload
0,0.00,0.000000,0.000000,0.000000,0,0.000000,0.000000,0.000000
1,0.01,0.000000,0.000000,0.000000,0,0.000000,0.000000,0.000000
2,0.02,0.000000,0.000000,0.000000,0,0.000000,0.000000,0.000000
3,0.03,0.000000,0.000000,0.000000,0,0.000000,0.000000,0.000000
4,0.04,0.000000,0.000000,0.000000,0,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...
596,5.96,90.013823,89.983617,90.014030,90,0.000643,0.007919,-0.000238
597,5.97,90.014069,89.990585,90.013975,90,0.000636,0.005816,-0.000243
598,5.98,90.014319,89.997399,90.013914,90,0.000623,0.003710,-0.000248
599,5.99,90.014572,90.003989,90.013850,90,0.000606,0.001626,-0.000253


### Rise-time, Overshoot and Steady-state Analysis

In [22]:
# Setpoint and range for rise time
setpoint = 90
lower = 0.1 * setpoint
upper = 0.9 * setpoint

PID = ['theta_PID1_noload', 'theta_PID2_noload', 'theta_PID3_noload']
results = {}
 
for pid_n in (PID):
    theta = noload[pid_n]
    
    # find the index of the first lower and upper thresholds value
    idx_start = theta.index[theta >= lower][0]
    idx_end   = theta.index[theta >= upper][0]
    
    rise_time = round(noload['time'].iloc[idx_end] - noload['time'].iloc[idx_start], 2)
    overshoot = round(theta.max() - setpoint, 2)
    overshoot_percent = round((overshoot/ setpoint) * 100,1)
    steady_state = round(setpoint - theta.iloc[-1], 2)

    results[pid_n] = {'Rise Time': rise_time,
                    'Overshoot': overshoot,
                    'Overshoot %': overshoot_percent,
                    'Steady-State Error': steady_state}

df_results = pd.DataFrame(results).T    # transpose the DataFrame
df_results.index.name = 'Case'
df_results.index = df_results.index.str.replace('theta_', '')
df_results.columns = ['Rise Time [s]', 'Overshoot [°]','Overshoot [%]', 
                      'Steady-State Error [°]']
df_results

,Rise Time [s],Overshoot [°],Overshoot [%],Steady-State Error [°]
Case,,,,
PID1_noload,0.12,50.32,55.9,-0.01
PID2_noload,0.12,69.63,77.4,-0.01
PID3_noload,0.13,38.21,42.5,-0.01


## CASE B: Load disturbance 
From the analysis of the no-load case, we compared the three PID controllers based on **Rise Time, Overshoot, and Steady-State Error**:

- **PID1**: moderate rise time, high overshoot;  
- **PID2**: similar rise time to PID1, but overshoot is even higher;  
- **PID3**: slightly longer rise time, but **significantly lower overshoot** and negligible steady-state error  

Based on this comparison, **PID3** is selected for the subsequent analysis with an **external load (disturbing torque)**.  

**Reasoning:**  
A low overshoot and minimal steady-state error are particularly important when a disturbing torque is applied, because a controller with high overshoot may cause oscillations and instability under load. PID3 offers a **stable and precise response**, making it the best candidate to handle the additional load while maintaining control accuracy.

#### Data Import

In [17]:
file_path_load = 'data/load_DC_motor.csv'

load = pd.read_csv(file_path_load)
load

,time,theta_PID3_noload,theta_PID3,theta_ref,Volt_PID3_noload,Volt_PID3,torque
0,0.00,0.000000,0.000000,0,0.000000,0.000000,0.0
1,0.01,0.000000,0.000000,0,0.000000,0.000000,0.0
2,0.02,0.000000,0.000000,0,0.000000,0.000000,0.0
3,0.03,0.000000,0.000000,0,0.000000,0.000000,0.0
4,0.04,0.000000,0.000000,0,0.000000,0.000000,0.0
...,...,...,...,...,...,...,...
596,5.96,90.014030,89.983194,90,-0.000238,7.200769,4.5
597,5.97,90.013975,89.983604,90,-0.000243,7.200758,4.5
598,5.98,90.013914,89.984009,90,-0.000248,7.200747,4.5
599,5.99,90.013850,89.984410,90,-0.000253,7.200735,4.5


In [27]:
PID_load = ['theta_PID3_noload', 'theta_PID3']
V_cols = ['Volt_PID3_noload', 'Volt_PID3']   
results_load = {}

for i,pid_load in enumerate(PID_load):
    theta_load = load[pid_load]  
    V = load[V_cols[i]]  

    # find the index of the first lower and upper thresholds value
    idx_start_load = theta_load.index[theta >= lower][0]
    idx_end_load   = theta_load.index[theta >= upper][0]
    
    rise_time_load = round(load['time'].iloc[idx_end] - load['time'].iloc[idx_start], 2)
    overshoot_load = round(theta_load.max() - setpoint, 2)
    overshoot_percent_load = round((overshoot_load/ setpoint) * 100,1)
    steady_state_err_load = round(setpoint - theta_load.iloc[-1], 2)

    # Voltage Steady-state values
    V_regime = V.iloc[int(len(V)*0.7):]  # Last 30% of the signal
    V_mean = round(V_regime.mean(), 2)
   
    results_load[pid_load] = {'Rise Time': rise_time_load,
                    'Overshoot': overshoot_load,
                    'Overshoot%': overshoot_percent_load,
                    'Steady-State Error': steady_state_err_load,
                    'Volt mean [V]': V_mean}


df_results_load = pd.DataFrame(results_load).T  # transpose the DataFrame
df_results_load.index.name = 'Case'
df_results_load.index = df_results_load.index.str.replace('theta_', '')
df_results_load.index = df_results_load.index.str.replace('theta_', '')
df_results_load.columns = ['Rise Time [s]', 'Overshoot [°]','Overshoot [%]', 
                           'Steady-State Error [°]','Mean steady-state voltage [V]']
df_results_load

,Rise Time [s],Overshoot [°],Overshoot [%],Steady-State Error [°],Mean steady-state voltage [V]
Case,,,,,
PID3_noload,0.13,38.21,42.5,-0.01,0.0
PID3,0.13,4.99,5.5,0.02,7.2


## Observations and conclusions

##### Rise Time  
Both Cases (PID3 wiht and without load) have the same rise time (0.13 s), so the load does not significantly affect the initial speed of the response.

##### Overshoot  
Without load, the overshoot is very high (38.21° / 42.5 %), while with load it is greatly reduced (4.99° / 5.5 %). This shows that the PID3 under load is more stable and less prone to initial spikes.

##### Steady-State Error  
Both cases have a negligible error (-0.01° without load, 0.02° with load), meaning the system correctly reaches the final setpoint.

##### Mean Voltage  
Without load the voltage is almost zero (0 V), while with load a mean voltage of 7.2 V is needed to maintain the desired position. This reflects the motor compensating for the load.

### Overall conclusion  
The PID works well in both cases, but the presence of load reduces overshoot, making the response more controlled. The applied voltage increases to compensate for the load, as expected.